# 03 Gold Tables (B): Products, sellers, calendar, reviews

**Owner:** Person B
**Builds:** `gold.dim_geography`, `gold.dim_products`, `gold.dim_sellers`, `gold.dim_date`, `gold.fact_reviews`, `gold.mart_seller_performance`, `gold.mart_daily_sales_weather`
**Rule:** joins happen only in gold, never in silver. Aggregate one-to-many tables to one row per key BEFORE joining.

## Build order
1. `dim_geography`
2. `dim_products`, `dim_sellers`
3. `dim_date`
4. `fact_reviews`
5. `mart_seller_performance` and `mart_daily_sales_weather` (need Person A's `fact_orders`, `fact_order_items`, `dim_customers`)

---

## 1. `gold.dim_geography`
- **Grain:** one row per `zip_code_prefix`.
- **Base:** `silver.geolocation`
- **Joins:** none (silver is already averaged to one row per prefix).
- **Columns:** `zip_code_prefix`, `lat`, `lng`, `city`, `state`.
- **Checks:** `zip_code_prefix` unique; row count equals `silver.geolocation`.

## 2. `gold.dim_products`
- **Grain:** one row per `product_id`.
- **Base:** `silver.products`
- **Joins:**

| Join | Type | Columns produced |
|---|---|---|
| `silver.category_translation` on `category_name` | `LEFT JOIN` | `category_name_english` |

- **Columns:** `product_id`, `category_name`, `category_name_english`, `name_length`, `description_length`, `photos_qty`, `weight_g`, `length_cm`, `height_cm`, `width_cm`.
- **Fallback:** if there is no translation, `category_name_english` = `category_name`; if both are NULL, use `'unknown'`.
- **Checks:** `product_id` unique; row count equals `silver.products` (a higher count means the translation table has duplicates); count of untranslated categories is reported.

## 3. `gold.dim_sellers`
- **Grain:** one row per `seller_id`.
- **Base:** `silver.sellers`
- **Joins:** none. `zip_code_prefix` stays as a key to `dim_geography`.
- **Columns:** `seller_id`, `zip_code_prefix`, `city`, `state`.
- **Checks:** `seller_id` unique; row count equals `silver.sellers`.

## 4. `gold.dim_date`
- **Grain:** one row per calendar day (2016-01-01 to 2018-12-31, covering the order dates).
- **Base:** generated calendar (recursive CTE or numbers table).
- **Calendar columns:** `date`, `year`, `quarter`, `month`, `month_name`, `week`, `day_of_week`, `is_weekend`.
- **Joins (all `LEFT JOIN` on `date`):**

| Join | Columns produced | Note |
|---|---|---|
| `silver.holidays` | `is_holiday`, `holiday_name` | `is_holiday` = 0 when there is no match |
| `silver.exchange_rates` | `usd_per_brl` | Fill gaps (weekends) with the last known rate; confirm the rule before coding |
| `silver.weather` | `sp_temp_max`, `sp_temp_min`, `sp_precipitation` | **São Paulo only.** The `sp_` prefix keeps this visible downstream |

- **Checks:** no missing days; row count equals the number of days in the range; each joined table adds no duplicate dates; weather columns are NULL outside the weather date range, not 0.
- **Power BI:** mark as the date table.

## 5. `gold.fact_reviews`
- **Grain:** one row per `order_id`.
- **Base:** `silver.reviews`
- **Joins:** none. Aggregate by `order_id` instead:
  - `AVG(review_score)` → `review_score_avg`
  - `MIN(review_score)` → `review_score_min`
  - `COUNT(*)` → `review_count`
  - `MIN(review_creation_date)` → `first_review_date`
- **Columns:** `order_id`, `review_score_avg`, `review_score_min`, `review_count`, `first_review_date`.
- **Checks:** `order_id` unique; every score is between 1 and 5; every `order_id` exists in `gold.fact_orders` (run after A finishes).

## 6. `gold.mart_seller_performance`
- **Grain:** one row per `seller_id`.
- **Base:** `gold.fact_order_items`
- **Joins:**

| Join | Type | Aggregate first | Purpose |
|---|---|---|---|
| `gold.fact_order_items` | base | one row per `seller_id` + `order_id` (`SUM(price)`, `COUNT(*)`) | seller revenue and items per order |
| `gold.fact_orders` on `order_id` | `LEFT JOIN` | none (already one row per order) | `delivery_days`, `is_late` |
| `gold.fact_reviews` on `order_id` | `LEFT JOIN` | none (already one row per order) | `review_score_avg` |
| `gold.dim_sellers` on `seller_id` | `LEFT JOIN` | none | `city`, `state` |

- **Columns:** `seller_id`, `city`, `state`, `orders`, `items_sold`, `revenue`, `avg_review_score`, `late_pct`, `avg_delivery_days`, `has_enough_orders` (1 if `orders` >= 10).
- **Logic:**
  - Aggregate items to one row per `seller_id` + `order_id` before joining to orders and reviews.
  - Count `late_pct` and `avg_delivery_days` only on delivered orders.
  - An order with several sellers is counted once for each of them, so the review and lateness are shared across those sellers. Note this in the log.
- **Why a mart:** the per-seller rollup mixes three fact tables, which is clumsy in DAX.
- **Checks:** `seller_id` unique; `SUM(revenue)` equals `SUM(price)` in `fact_order_items`; `avg_review_score` is between 1 and 5.

## 7. `gold.mart_daily_sales_weather`
- **Grain:** one row per day, **São Paulo customers only** (`state = 'SP'`).
- **Base:** `gold.dim_date`
- **Joins:**

| Join | Type | Aggregate first | Purpose |
|---|---|---|---|
| `gold.fact_orders` + `gold.dim_customers` (filter `state = 'SP'`) by `order_date` | `LEFT JOIN` | `COUNT(*)` orders per day | `orders` |
| `gold.fact_order_items` on `order_id` | `LEFT JOIN` | `SUM(price)` per `order_id`, then per day | `revenue` |
| `gold.fact_orders` + `gold.dim_customers` (SP) by `delivered_customer_date` | `LEFT JOIN` | `COUNT(*)` and `SUM(is_late)` per delivered day | `delivered_orders`, `late_orders` |
| `silver.weather` (already in `dim_date` as `sp_*`) | none | none | `sp_precipitation`, `sp_temp_max`, `sp_temp_min` |

- **Columns:** `date`, `orders`, `revenue`, `delivered_orders`, `late_orders`, `late_pct`, `sp_precipitation`, `sp_temp_max`, `sp_temp_min`, `is_rainy` (precipitation above the chosen threshold), `is_holiday`, `is_weekend`.
- **Logic:**
  - Orders and revenue are grouped by the **purchase date**; late deliveries are grouped by the **delivered date**, because weather matters when the package is on the road.
  - Aggregate each source to one row per day in its own subquery, then join them to `dim_date`, so the sources do not multiply each other.
  - Days without orders get 0, not NULL. Days without weather stay NULL.
- **Limitation:** weather covers São Paulo only. Mark every visual that uses this mart "SP only".
- **Checks:** `date` unique; `SUM(orders)` equals the number of SP orders in `fact_orders`; `SUM(revenue)` equals SP item revenue.

## Validation (run at the end)
- Row counts: gold vs silver for every table above.
- No duplicate keys.
- Orphan checks: every `fact_order_items.product_id` exists in `dim_products` and every `seller_id` exists in `dim_sellers`.

In [1]:
import pandas as pd
from sqlalchemy import text
import sys

# db.py faylının yerləşdiyi qovluğu əlavə edirik
sys.path.append("/workspaces/olist-ecommerce-analytics/notebooks")

from db import get_engine

engine = get_engine()

## 1. `gold.dim_geography`

#burada silver.geolocation cədvəlinin sətr sayları ilə poct indekslərinin unikal sayına baxırıq

In [2]:
df = pd.read_sql("""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT zip_code_prefix) AS unique_prefixes
    FROM silver.geolocation
""", engine)

df

,total_rows,unique_prefixes
0,1000163,19015


In [3]:
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.geolocation
""", engine)

df

,zip_code_prefix,lat,lng,city,state
0,4206,-23.594713,-46.603403,sao paulo,SP
1,4202,-23.578235,-46.604193,sao paulo,SP
2,4212,-23.589079,-46.601745,sao paulo,SP
3,4216,-23.598318,-46.604131,sao paulo,SP
4,4279,-23.601854,-46.604879,sao paulo,SP


#Unikal poct prefiksi üçün bir sətrdə digər sütunları qruplaşdırırıq təkrarlanan məlumatları ümümləşdiririk

In [4]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_geography"))

    # Hər poçt prefiksi üçün bir sətir yaradırıq
    conn.execute(text("""
        SELECT
            zip_code_prefix,       -- Poçt prefiksi
            AVG(lat) AS lat,        -- Orta enlik koordinatı
            AVG(lng) AS lng,        -- Orta uzunluq koordinatı
            MAX(city) AS city,      -- Şəhər adı
            MAX(state) AS state     -- Ştat kodu
        INTO gold.dim_geography    -- Yeni Gold cədvəli yaradırıq
        FROM silver.geolocation   
        GROUP BY zip_code_prefix  
    """))

In [5]:
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT zip_code_prefix) AS unique_prefixes
    FROM gold.dim_geography
""", engine)

print(df.to_string(index=False))

 total_rows  unique_prefixes
      19015            19015


## 2. `gold.dim_products`

In [6]:
# Məhsullar cədvəlindəki sütunlara baxırıq
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.products
""", engine)
df

,product_id,category_name,name_length,description_length,photos_qty,weight_g,length_cm,height_cm,width_cm
0,00066f42aeeb9f3007548bb9d3f33c38,perfumaria,53,596,6,300.0,20.0,16.0,16.0
1,00088930e925c41fd95ebfe695fd2655,automotivo,56,752,4,1225.0,55.0,10.0,26.0
2,0009406fd7479715e4bef61dd91f2462,cama_mesa_banho,50,266,2,300.0,45.0,15.0,35.0
3,001795ec6f1b187d37335e1c4704762e,consoles_games,53,274,1,600.0,30.0,20.0,20.0
4,001b237c0e9bb435f2e54071129237e9,cama_mesa_banho,42,253,1,6000.0,40.0,4.0,30.0


In [7]:
# Kateqoriya tərcümələri cədvəlindəki sütunlara baxırıq
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.category_translation
""", engine)

df

,category_name,category_name_english
0,agro_industria_e_comercio,agro_industry_and_commerce
1,alimentos,food
2,alimentos_bebidas,food_drink
3,artes,art
4,artes_e_artesanato,arts_and_craftmanship


Burada məhsul məlumatları ilə tərcümə olunmuş kateqoriya adlarını birləşdirib bir cədvələ yığırıq

In [8]:

# Məhsullar və kateqoriya tərcümələrini birləşdiririk
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_products"))

    conn.execute(text("""
        SELECT
            p.product_id,
            p.category_name,
            COALESCE(t.category_name_english, p.category_name, 'unknown') AS category_name_english,
            p.name_length,
            p.description_length,
            p.photos_qty,
            p.weight_g,
            p.length_cm,
            p.height_cm,
            p.width_cm
        INTO gold.dim_products
        FROM silver.products p
        LEFT JOIN silver.category_translation t
            ON p.category_name = t.category_name
    """))

In [9]:
# Sətir sayını və təkrarlanan məhsul ID-lərini yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT product_id) AS unique_products
    FROM gold.dim_products
""", engine)

df

,total_rows,unique_products
0,32951,32951


## 3. `gold.dim_sellers`

In [10]:
# Satıcılar cədvəlini yoxlayırıq
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.sellers
""", engine)

df

,seller_id,zip_code_prefix,city,state
0,0015a82c2db000af6aaaf3ae2ecb0532,9080,santo andre,SP
1,001cca7ae9ae17fb1caed9dfb1094831,29156,cariacica,ES
2,001e6ad469a905060d959994f1b41e4f,24754,sao goncalo,RJ
3,002100f778ceb8431b7a1020ff7ab48f,14405,franca,SP
4,010da0602d7774602cd1b3f5fb7b709e,9710,sao bernardo do campo,SP


In [11]:
# Satıcı məlumatlarını Gold cədvəlinə köçürürük
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_sellers"))

    conn.execute(text("""
        SELECT
            seller_id,
            zip_code_prefix,
            city,
            state
        INTO gold.dim_sellers
        FROM silver.sellers
    """))

In [12]:
# Satıcı sayını və təkrarlanan ID-ləri yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT seller_id) AS unique_sellers
    FROM gold.dim_sellers
""", engine)

df

,total_rows,unique_sellers
0,3095,3095


## 4. `gold.dim_date`

In [13]:
# Tarixlər cədvəlini yaradırıq
with engine.begin() as conn:  # Verilənlər bazasına qoşuluruq və əməliyyatları idarə edirik
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_date"))  # Əvvəlki cədvəl varsa, silirik

    conn.execute(text("""
        WITH dates AS (  -- Tarixləri ardıcıl yaratmaq üçün müvəqqəti sorğu hazırlayırıq
            SELECT CAST('2016-01-01' AS DATE) AS [date]  -- Başlanğıc tarixini 1 yanvar 2016 təyin edirik

            UNION ALL  -- Yeni tarixləri əvvəlki nəticəyə əlavə edirik

            SELECT DATEADD(DAY, 1, [date])  -- Əvvəlki tarixə 1 gün əlavə edirik
            FROM dates  -- Əvvəl yaratdığımız tarixlərdən istifadə edirik
            WHERE [date] < '2018-12-31'  -- Son tarixə çatana qədər davam edirik
        )
        SELECT
            [date],  -- Tarixi göstəririk
            YEAR([date]) AS [year],  -- Tarixin ilini çıxarırıq
            DATEPART(QUARTER, [date]) AS [quarter],  -- İlin hansı rübü olduğunu tapırıq
            MONTH([date]) AS [month],  -- Ayın nömrəsini çıxarırıq
            DATENAME(MONTH, [date]) AS month_name,  -- Ayın adını çıxarırıq
            DATEPART(ISO_WEEK, [date]) AS [week],  -- ISO standartına görə həftənin nömrəsini tapırıq
            ((DATEPART(WEEKDAY, [date]) + @@DATEFIRST - 2) % 7) + 1 AS day_of_week,  -- Həftənin gününü bazar ertəsi=1, bazar=7 formasına gətiririk
            CASE
                WHEN ((DATEPART(WEEKDAY, [date]) + @@DATEFIRST - 2) % 7) + 1 IN (6, 7)  -- Gün şənbə və ya bazardırsa
                THEN 1 ELSE 0  -- Həftəsonudursa 1, deyilsə 0 yazırıq
            END AS is_weekend  -- Bu sütunun adını is_weekend qoyuruq
        INTO gold.dim_date  -- Nəticəni gold.dim_date adlı yeni cədvələ yazırıq
        FROM dates  -- Yaratdığımız tarix siyahısından məlumat götürürük
        OPTION (MAXRECURSION 0)  -- Təkrarlanan sorğu üçün maksimum addım məhdudiyyətini götürürük
    """))

In [14]:
pd.read_sql(
    "SELECT COUNT(*) AS total_dates FROM gold.dim_date",
    engine
)

,total_dates
0,1096


In [15]:
# Bayram günlərini göstərən sütunu cədvələ əlavə edirik
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE gold.dim_date
        ADD is_holiday BIT NULL
    """))

In [16]:
# Bayram cədvəlindən ilk 5 sətri oxuyuruq
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.holidays
""", engine)

df

,holiday_date,local_name,name,is_national,state_code,types
0,2016-01-01,Confraternização Universal,New Year's Day,True,None,"[""Public""]"
1,2016-02-08,Carnaval,Carnival,True,None,"[""Bank"", ""Optional""]"
2,2016-02-09,Carnaval,Carnival,True,None,"[""Bank"", ""Optional""]"
3,2016-03-25,Sexta-feira Santa,Good Friday,True,None,"[""Public""]"
4,2016-03-27,Domingo de Páscoa,Easter Sunday,True,None,"[""Public""]"


In [17]:
# Milli bayramları tarix cədvəli ilə uyğunlaşdırırıq
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE d
        SET is_holiday = CASE
            WHEN h.holiday_date IS NOT NULL THEN 1
            ELSE 0
        END
        FROM gold.dim_date d
        LEFT JOIN silver.holidays h
            ON d.[date] = h.holiday_date
            AND h.is_national = 1
    """))

In [18]:
# Tarixlərin və milli bayramların sayını yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_dates,
        COUNT(DISTINCT [date]) AS unique_dates,
        SUM(CASE WHEN is_weekend = 1 THEN 1 ELSE 0 END) AS weekend_days,
        SUM(CASE WHEN is_holiday = 1 THEN 1 ELSE 0 END) AS holidays
    FROM gold.dim_date
""", engine)

df

,total_dates,unique_dates,weekend_days,holidays
0,1096,1096,314,39


In [19]:
# Tarix sayını və tarix aralığını yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT [date]) AS unique_dates,
        MIN([date]) AS first_date,
        MAX([date]) AS last_date
    FROM gold.dim_date
""", engine)

display(df)

,total_rows,unique_dates,first_date,last_date
0,1096,1096,2016-01-01,2018-12-31


In [20]:
# Bayramlar cədvəlinə baxırıq
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.holidays
""", engine)

display(df)

,holiday_date,local_name,name,is_national,state_code,types
0,2016-01-01,Confraternização Universal,New Year's Day,True,None,"[""Public""]"
1,2016-02-08,Carnaval,Carnival,True,None,"[""Bank"", ""Optional""]"
2,2016-02-09,Carnaval,Carnival,True,None,"[""Bank"", ""Optional""]"
3,2016-03-25,Sexta-feira Santa,Good Friday,True,None,"[""Public""]"
4,2016-03-27,Domingo de Páscoa,Easter Sunday,True,None,"[""Public""]"


In [21]:
# Bayram günlərinin sayını yoxlayırıq
df = pd.read_sql("""
    SELECT
        SUM(CASE WHEN is_holiday = 1 THEN 1 ELSE 0 END) AS holiday_days,
        COUNT(*) AS total_days
    FROM gold.dim_date
""", engine)

display(df)

,holiday_days,total_days
0,39,1096


In [22]:
# Valyuta məzənnələri cədvəlindən ilk 5 sətri göstəririk
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.exchange_rates
""", engine)

df

,rate_date,rate_usd
0,2015-12-31,0.25250
1,2016-01-04,0.24755
2,2016-01-05,0.24977
3,2016-01-06,0.24791
4,2016-01-07,0.24699


In [23]:
# Dollar məzənnəsini saxlamaq üçün sütun əlavə edirik
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE gold.dim_date
        ADD usd_per_brl DECIMAL(10,5) NULL
    """))

In [24]:
# Hər tarix üçün həmin günə və ya əvvəlki ən yaxın məzənnəni götürürük
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE d
        SET usd_per_brl = (
            SELECT TOP 1 e.rate_usd
            FROM silver.exchange_rates e
            WHERE e.rate_date <= d.[date]
            ORDER BY e.rate_date DESC
        )
        FROM gold.dim_date d
    """))

In [25]:
# Məzənnələrin düzgün yazıldığını yoxlayırıq
df = pd.read_sql("""
    SELECT TOP 10 [date], usd_per_brl
    FROM gold.dim_date
    ORDER BY [date]
""", engine)

df

,date,usd_per_brl
0,2016-01-01,0.25250
1,2016-01-02,0.25250
2,2016-01-03,0.25250
3,2016-01-04,0.24755
4,2016-01-05,0.24977
5,2016-01-06,0.24791
6,2016-01-07,0.24699
7,2016-01-08,0.24862
8,2016-01-09,0.24862
9,2016-01-10,0.24862


In [26]:
# Hava məlumatları cədvəlinin sütunlarını və ilk 5 sətrini yoxlayırıq
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM silver.weather
""", engine)

df

,weather_date,temp_max,temp_min,precipitation
0,2016-01-01,27.7,21.0,9.1
1,2016-01-02,24.5,19.8,25.7
2,2016-01-03,23.2,18.9,4.4
3,2016-01-04,25.4,17.7,0.0
4,2016-01-05,26.4,16.8,0.0


In [27]:
# Hava məlumatlarını saxlamaq üçün 3 sütun əlavə edirik
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE gold.dim_date
        ADD sp_temp_max FLOAT NULL,
            sp_temp_min FLOAT NULL,
            sp_precipitation FLOAT NULL
    """))

In [28]:
# Hava məlumatlarını tarixlərə uyğunlaşdırıb Gold cədvəlinə yazırıq
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE d
        SET
            sp_temp_max = w.temp_max,
            sp_temp_min = w.temp_min,
            sp_precipitation = w.precipitation
        FROM gold.dim_date d
        LEFT JOIN silver.weather w
            ON d.[date] = w.weather_date
    """))

In [29]:
# Tarix, bayram, məzənnə və hava məlumatlarını yoxlayırıq
df = pd.read_sql("""
    SELECT TOP 10
        [date],
        is_weekend,
        is_holiday,
        usd_per_brl,
        sp_temp_max,
        sp_temp_min,
        sp_precipitation
    FROM gold.dim_date
    ORDER BY [date]
""", engine)

df

,date,is_weekend,is_holiday,usd_per_brl,sp_temp_max,sp_temp_min,sp_precipitation
0,2016-01-01,0,True,0.25250,27.7,21.0,9.1
1,2016-01-02,1,False,0.25250,24.5,19.8,25.7
2,2016-01-03,1,False,0.25250,23.2,18.9,4.4
3,2016-01-04,0,False,0.24755,25.4,17.7,0.0
4,2016-01-05,0,False,0.24977,26.4,16.8,0.0
5,2016-01-06,0,False,0.24791,28.5,18.5,0.0
6,2016-01-07,0,False,0.24699,28.6,18.6,0.3
7,2016-01-08,0,False,0.24862,28.4,20.0,0.9
8,2016-01-09,1,False,0.24862,28.9,20.4,30.5
9,2016-01-10,1,False,0.24862,27.4,21.5,14.2


## 5. `gold.fact_reviews`

In [30]:
# Rəylər cədvəlinin sütunlarını və ilk sətirlərini yoxlayırıq
df = pd.read_sql("SELECT TOP 5 * FROM silver.reviews", engine)
df

,review_id,order_id,review_score,comment_title,comment_message,creation_date,answer_timestamp
0,00020c7512a52e92212f12d3e37513c0,e28abf2eb2f1fbcbdc2dd0cd9a561671,5,Entrega rápida!,A entrega foi super rápida e o pendente é lind...,2018-04-25,2018-04-26 14:55:36
1,00032b0141443497c898b3093690af51,04fb47576993a3cb0c12d4b25eab6e4e,5,None,None,2017-05-30,2017-06-01 23:28:55
2,000359bceee1b6fe876b30d35b5f9ef2,97868e4c884bc85f4501ae0a66181cc9,5,None,None,2018-02-06,2018-02-06 20:54:37
3,0005534973388c830bb858cfba83b17b,a589caa6892ceacc6bbf2f8cc30a8ad4,5,None,otimo produto. prazo cumprido. sabor tambem mu...,2017-09-01,2017-09-02 13:04:08
4,00055e36e9608fe969231e551983a69c,f5fea26ab547eec920e6f8ecdc5c37e4,5,None,O produto foi entregue muito antes do esperado...,2017-11-25,2017-11-27 11:37:22


In [31]:
# Rəylər cədvəlinin məlumat keyfiyyətini yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT review_id) AS unique_reviews,
        SUM(CASE WHEN review_id IS NULL THEN 1 ELSE 0 END) AS null_review_id,
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
        SUM(CASE WHEN review_score IS NULL THEN 1 ELSE 0 END) AS null_review_score,
        MIN(review_score) AS min_score,
        MAX(review_score) AS max_score
    FROM silver.reviews
""", engine)

display(df)

,total_rows,unique_reviews,null_review_id,null_order_id,null_review_score,min_score,max_score
0,99224,98410,0,0,0,1,5


In [32]:
# Rəy məlumatlarını Gold fakt cədvəlinə köçürürük
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.fact_reviews"))

    conn.execute(text("""
        SELECT
            r.review_id,
            r.order_id,
            r.review_score,
            r.comment_title,
            r.comment_message,
            r.creation_date,
            r.answer_timestamp
        INTO gold.fact_reviews
        FROM silver.reviews r
        INNER JOIN silver.orders o
            ON r.order_id = o.order_id
    """))

In [33]:
# Gold fakt cədvəlinin sətir sayını yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT CONCAT(review_id, order_id)) AS unique_combinations
    FROM gold.fact_reviews
""", engine)

display(df)

,total_rows,unique_combinations
0,99224,99224


## 6. `gold.mart_seller_performance`

In [34]:
# Sifariş məhsulları cədvəlinin ilk sətirlərinə baxırıq
df = pd.read_sql("SELECT TOP 5 * FROM silver.order_items", engine)
display(df)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [35]:
# Sifariş məhsullarının sayını və əsas boş dəyərləri yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS unique_orders,
        COUNT(DISTINCT seller_id) AS unique_sellers,
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
        SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END) AS null_seller_id,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price,
        SUM(CASE WHEN freight_value IS NULL THEN 1 ELSE 0 END) AS null_freight
    FROM silver.order_items
""", engine)

df

,total_rows,unique_orders,unique_sellers,null_order_id,null_seller_id,null_price,null_freight
0,112650,98666,3095,0,0,0,0


In [36]:
# Sifariş statuslarının sayını yoxlayırıq
df = pd.read_sql("""
    SELECT
        order_status,
        COUNT(*) AS total_orders
    FROM silver.orders
    GROUP BY order_status
    ORDER BY total_orders DESC
""", engine)

display(df)

,order_status,total_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [37]:
# Satıcıların çatdırılmış sifarişlər üzrə göstəricilərini hesablayırıq
df = pd.read_sql("""
    SELECT
        oi.seller_id,
        COUNT(DISTINCT oi.order_id) AS total_orders,
        COUNT(*) AS items_sold,
        SUM(oi.price) AS total_sales,
        SUM(oi.freight_value) AS total_freight
    FROM silver.order_items oi
    INNER JOIN silver.orders o
        ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY oi.seller_id
    ORDER BY total_sales DESC
""", engine)

df.head(10)

,seller_id,total_orders,items_sold,total_sales,total_freight
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,1148,226987.93,20019.13
1,53243585a1d6dc2643021fd1853d8905,348,400,217940.44,12856.58
2,4a3ca9315b744ce9f8e9374361493884,1772,1949,196882.12,34338.31
3,fa1c13f2614d7b5c4749cbc52fecda94,578,579,190917.14,9916.36
4,7c67e1448b00f6e969d365cea6b010ab,973,1355,186570.05,51236.64
5,7e93a43ef30c4f03f38b393420bc753a,319,322,165981.49,5992.06
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,1548,159816.87,24889.91
7,7a67c85e85bb2ce8582c35f2203ad736,1145,1155,139658.69,20619.83
8,1025f0e2d44d7041d6cf58b6550e0bfa,910,1420,138208.56,33716.40
9,955fee9216a65b617aa5c0531780ce60,1261,1472,131836.71,24769.77


In [38]:
# Satıcı performansı mart cədvəlini yaradırıq
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.mart_seller_performance"))

    conn.execute(text("""
        SELECT
            s.seller_id,
            s.zip_code_prefix,
            s.city,
            s.state,
            COALESCE(x.total_orders, 0) AS total_orders,
            COALESCE(x.items_sold, 0) AS items_sold,
            COALESCE(x.total_sales, 0) AS total_sales,
            COALESCE(x.total_freight, 0) AS total_freight
        INTO gold.mart_seller_performance
        FROM silver.sellers s
        LEFT JOIN (
            SELECT
                oi.seller_id,
                COUNT(DISTINCT oi.order_id) AS total_orders,
                COUNT(*) AS items_sold,
                SUM(oi.price) AS total_sales,
                SUM(oi.freight_value) AS total_freight
            FROM silver.order_items oi
            INNER JOIN silver.orders o
                ON oi.order_id = o.order_id
            WHERE o.order_status = 'delivered'
            GROUP BY oi.seller_id
        ) x
            ON s.seller_id = x.seller_id
    """))

In [39]:
# Yeni mart cədvəlinin sətir sayını və satıcı unikallığını yoxlayırıq
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT seller_id) AS unique_sellers,
        SUM(total_orders) AS total_orders,
        SUM(items_sold) AS items_sold,
        SUM(total_sales) AS total_sales
    FROM gold.mart_seller_performance
""", engine)

display(df)

,total_rows,unique_sellers,total_orders,items_sold,total_sales
0,3095,3095,97819,110197,13221498.11


## 7. `gold.mart_daily_sales_weather`

In [40]:
# Sifarişlərin tarix və status məlumatlarına baxırıq
df = pd.read_sql("""
    SELECT TOP 5
        order_id,
        order_status,
        order_purchase_timestamp
    FROM silver.orders
    ORDER BY order_purchase_timestamp
""", engine)

display(df)

,order_id,order_status,order_purchase_timestamp
0,2e7a8482f6fb09756ca50c10d7bfc047,shipped,2016-09-04 21:15:19
1,e5fa5a7210941f7d56d0208e4e071d35,canceled,2016-09-05 00:15:34
2,809a282bbd5dbcabb6f2f724fca862ec,canceled,2016-09-13 15:24:19
3,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38
4,71303d7e93b399f5bcd537d124c0bcfa,canceled,2016-10-02 22:07:52


In [41]:
# Çatdırılmış sifarişlər üzrə gündəlik satışları hesablayırıq
df = pd.read_sql("""
    SELECT
        CAST(o.order_purchase_timestamp AS DATE) AS sales_date,
        COUNT(DISTINCT o.order_id) AS total_orders,
        COUNT(*) AS items_sold,
        SUM(oi.price) AS total_sales,
        SUM(oi.freight_value) AS total_freight
    FROM silver.orders o
    INNER JOIN silver.order_items oi
        ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY CAST(o.order_purchase_timestamp AS DATE)
    ORDER BY sales_date
""", engine)

display(df.head())

,sales_date,total_orders,items_sold,total_sales,total_freight
0,2016-09-15,1,3,134.97,8.49
1,2016-10-03,7,7,441.98,117.55
2,2016-10-04,54,63,8595.89,1225.53
3,2016-10-05,35,48,6169.77,1039.73
4,2016-10-06,41,47,5889.96,908.94


In [42]:
# Gündəlik satışları tarix və hava məlumatları ilə birləşdiririk
df = pd.read_sql("""
    SELECT
        d.[date] AS sales_date,
        d.[year],
        d.[month],
        d.is_weekend,
        d.is_holiday,
        d.usd_per_brl,
        d.sp_temp_max,
        d.sp_temp_min,
        d.sp_precipitation,
        COALESCE(s.total_orders, 0) AS total_orders,
        COALESCE(s.items_sold, 0) AS items_sold,
        COALESCE(s.total_sales, 0) AS total_sales,
        COALESCE(s.total_freight, 0) AS total_freight
    FROM gold.dim_date d
    LEFT JOIN (
        SELECT
            CAST(o.order_purchase_timestamp AS DATE) AS sales_date,
            COUNT(DISTINCT o.order_id) AS total_orders,
            COUNT(*) AS items_sold,
            SUM(oi.price) AS total_sales,
            SUM(oi.freight_value) AS total_freight
        FROM silver.orders o
        INNER JOIN silver.order_items oi
            ON o.order_id = oi.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY CAST(o.order_purchase_timestamp AS DATE)
    ) s
        ON d.[date] = s.sales_date
    ORDER BY d.[date]
""", engine)

df.head()

,sales_date,year,month,is_weekend,is_holiday,usd_per_brl,sp_temp_max,sp_temp_min,sp_precipitation,total_orders,items_sold,total_sales,total_freight
0,2016-01-01,2016,1,0,True,0.25250,27.7,21.0,9.1,0,0,0.0,0.0
1,2016-01-02,2016,1,1,False,0.25250,24.5,19.8,25.7,0,0,0.0,0.0
2,2016-01-03,2016,1,1,False,0.25250,23.2,18.9,4.4,0,0,0.0,0.0
3,2016-01-04,2016,1,0,False,0.24755,25.4,17.7,0.0,0,0,0.0,0.0
4,2016-01-05,2016,1,0,False,0.24977,26.4,16.8,0.0,0,0,0.0,0.0


In [43]:
# Gündəlik satış və hava məlumatlarını Gold mart cədvəlinə yazırıq
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.mart_daily_sales_weather"))

    conn.execute(text("""
        SELECT
            d.[date] AS sales_date,
            d.[year],
            d.[month],
            d.is_weekend,
            d.is_holiday,
            d.usd_per_brl,
            d.sp_temp_max,
            d.sp_temp_min,
            d.sp_precipitation,
            COALESCE(s.total_orders, 0) AS total_orders,
            COALESCE(s.items_sold, 0) AS items_sold,
            COALESCE(s.total_sales, 0) AS total_sales,
            COALESCE(s.total_freight, 0) AS total_freight
        INTO gold.mart_daily_sales_weather
        FROM gold.dim_date d
        LEFT JOIN (
            SELECT
                CAST(o.order_purchase_timestamp AS DATE) AS sales_date,
                COUNT(DISTINCT o.order_id) AS total_orders,
                COUNT(*) AS items_sold,
                SUM(oi.price) AS total_sales,
                SUM(oi.freight_value) AS total_freight
            FROM silver.orders o
            INNER JOIN silver.order_items oi
                ON o.order_id = oi.order_id
            WHERE o.order_status = 'delivered'
            GROUP BY CAST(o.order_purchase_timestamp AS DATE)
        ) s
            ON d.[date] = s.sales_date
    """))

In [44]:
# Yaradılmış mart cədvəlinin ilk 5 sətrini göstəririk
df = pd.read_sql("""
    SELECT TOP 5 *
    FROM gold.mart_daily_sales_weather
    ORDER BY sales_date
""", engine)

df

,sales_date,year,month,is_weekend,is_holiday,usd_per_brl,sp_temp_max,sp_temp_min,sp_precipitation,total_orders,items_sold,total_sales,total_freight
0,2016-01-01,2016,1,0,True,0.25250,27.7,21.0,9.1,0,0,0.0,0.0
1,2016-01-02,2016,1,1,False,0.25250,24.5,19.8,25.7,0,0,0.0,0.0
2,2016-01-03,2016,1,1,False,0.25250,23.2,18.9,4.4,0,0,0.0,0.0
3,2016-01-04,2016,1,0,False,0.24755,25.4,17.7,0.0,0,0,0.0,0.0
4,2016-01-05,2016,1,0,False,0.24977,26.4,16.8,0.0,0,0,0.0,0.0
